# Données enrichies et optimisation économique

Comparaison à M0 et au QRF de référence sur les mêmes cas. Nouvelles variables NASA, humidité/évapotranspiration SIM, sols et pondération agricole ; distributions plus détaillées, dépendance blé–légumineuse et sélection temporelle des réglages. 20 000 scénarios par cas. Les hypothèses de chaque expérience et les sources sont archivées.

CPU suffisant. Aucune clé TabPFN requise. Sauvegarde Drive après chaque culture–modèle–année, reprise automatique. Les cartes de sol et réanalyses actuelles définissent une expérience rétrospective ; la sensibilité 2023–2025 est publiée séparément.

La suite exécute 34 configurations principales, puis deux variantes météo pondérées par le RPG 2015. Les 36 configurations sont comparées après assemblage contrôlé des sources.


In [ ]:
from pathlib import Path
import io, requests, zipfile
REPOSITORY = 'nassim-arifette/hackthefork-hackathon'
REF = 'research/data-model-optimization'  # Remplacer par un SHA pour figer une révision.
response = requests.get(f'https://api.github.com/repos/{REPOSITORY}/zipball/{REF}', timeout=120)
response.raise_for_status()
source = Path('/content/hackthefork-advanced-source')
source.mkdir(exist_ok=True)
with zipfile.ZipFile(io.BytesIO(response.content)) as archive:
    for entry in archive.infolist():
        if not (source / entry.filename).resolve().is_relative_to(source.resolve()):
            raise ValueError('Chemin d’archive invalide')
    archive.extractall(source)
    project = source / archive.namelist()[0].split('/')[0]
print('Sources :', project)


In [ ]:
import subprocess, sys
subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', '-r', str(project / 'requirements-research.txt')])


In [ ]:
from google.colab import drive
drive.mount('/content/drive')


In [ ]:
import runpy
runpy.run_path(str(project / 'scripts/colab_advanced.py'), run_name='__main__')


Le dossier contient les sources exécutées, les versions de dépendances, les paramètres, les points de reprise et les résultats. Le fichier `progress.json` décrit la dernière étape. Une modification des sources ou des données ouvre un dossier distinct.


### Reproductibilité du calcul QRF

Le lanceur fixe le profil NumPy utilisé dans les expériences (`NPY_DISABLE_CPU_FEATURES` pour AVX512), avant de démarrer les processus de calcul. Cela stabilise l’ordre des ex æquo avant l’échantillonnage des feuilles QRF entre les machines testées. Le réglage est enregistré dans `execution_environment.json` et entre dans la clé de reprise via le code du lanceur.
